In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import matplotlib
import matplotlib.pyplot as plt
import sigpy as sp
import pandas as pd
import sigpy.plot as pl
import numpy as np
import os
import sys
from pathlib import Path
import jax as jx
import time
import h5py

#ksp = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_ksp.npy")
#coord = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_coord.npy")

sys.path.insert(0,'/Users/ayman/Desktop/MSc Project Local/MSc-Project-ZTE')
import aymansigmri as asm


sys.path.insert(0, "/Users/ayman/Documents/GitHub/riesling/python")

import riesling as rlp



riesling_bin = Path.home() / "Documents" / "github" / "riesling" / "build" / "cxx" / "riesling"
os.environ["PATH"] += os.pathsep + str(riesling_bin)

data = '/Users/ayman/Desktop/MSc Project Local/MSc-Project-ZTE/riselingwork/hires-nufft'

In [ ]:
riesling_im, _ = asm.h5tonum(f'{data}.h5', showshape=True)
#riesling_cartksp = sp.fft(riesling_im)


## Goal of the notebook

### Integrate riesling into sigpy pipeline

1. Start with NUFFT inverse

- get the above file format into something that riseling can read

In [ ]:
with h5py.File(f'{data}.h5', 'r') as src, h5py.File(f'{data}-notraj.h5', 'w') as dst:
    src.copy('data', dst)
    # copy any other datasets you need, but not 'trajectory'
    for k, v in src.attrs.items():
        dst.attrs[k] = v

In [ ]:
def num_to_h5_cart(arr, outfile):
    arr = np.asarray(arr, dtype=np.complex64)
    print(arr.shape)
    with h5py.File(outfile, 'w') as f:
        f.create_dataset('data', data=arr)

num_to_h5_cart(riesling_cartksp, f'{data}-test.h5')

In [ ]:
dim = 256

#!riesling op-pad "{data}-test.h5" "{data}-pad.h5" {dim},{dim},{dim} -f -v3

In [ ]:
enlarged_cartesian = np.load(f"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3D Data/enlarged_cartesian.npy")
#riesling_enlargedcart = asm.h5tonum(f'{data}-pad.h5', showshape=True, load_traj=False, stripdims=False)

In [ ]:
asm.plot_planes(data=enlarged_cartesian, title=None, inputtype_kspace=True)

In [ ]:
width=4
gap = 3
num_iters = 50


inner_wid=26
indiv2 = int(inner_wid/2)



cy = cx = cz = inner_wid // 2
r = 3
yy, xx, zz = np.ogrid[:inner_wid, :inner_wid, :inner_wid]
mask = (yy - cy)**2 + (xx - cx)**2 + (zz - cz)**2 > r**2 + 2




innersidelen=inner_wid
sidelencart = 13
sidelenrad = 7

In [ ]:


#inner_region, inner_mask, start, end = inner_portion(enlarged_kspace=enlarged_cartesian, inner_sidelen=inner_wid)
#np.savez(f"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3d Data/inner_cache.npy", inner_region=inner_region, start=start, end=end)


In [ ]:
def hankel(kspace, w):
    N_c = kspace.shape[0]
    N_x = kspace.shape[1] - w + 1
    N_y = kspace.shape[2] - w + 1
    N_z = kspace.shape[3] - w + 1
    data_matrix = np.empty((N_c * w*w*w, N_x*N_y*N_z), dtype=np.complex64)
    for c in range(N_c):
        onecoil_matrix = data_matrix[c*w*w*w:(c+1)*w*w*w]
        col_index = 0
        for k in range(N_z):
            for i in range(N_y):
                for j in range(N_x):
                    mat = kspace[c, j:j+w, i:i+w, k:k+w]
                    onecoil_matrix[:, col_index] = mat.reshape(-1, order='F')
                    col_index += 1
    return data_matrix, N_c, N_x, N_y, N_z


def hankel_H_averaged(data_matrix, n_coils, Nx, Ny, Nz, w=3):
    kspace = np.empty((n_coils, Nx, Ny, Nz), dtype=np.complex64)
    split_arr = np.array(np.split(data_matrix, n_coils))

    n_win_x = Nx - w + 1
    n_win_y = Ny - w + 1
    n_win_z = Nz - w + 1

    for c in range(n_coils):
        singlecoil = split_arr[c]
        transposed = singlecoil.T

        recon = np.zeros((Nx, Ny, Nz), dtype=np.complex64)
        counts = np.zeros((Nx, Ny, Nz), dtype=np.float32)

        for index in range(len(transposed)):
            win = transposed[index].reshape((w, w, w), order='F')

            j = index % n_win_x
            i = (index // n_win_x) % n_win_y
            k = index // (n_win_x * n_win_y)

            recon[j:j+w, i:i+w, k:k+w] += win
            counts[j:j+w, i:i+w, k:k+w] += 1

        kspace[c] = recon / counts
    return kspace


def zeropadding3d(data, im_dim, input_kspace=True, output_kspace=True):
    n_coils = data.shape[0]
    print('starting resize')
    if input_kspace:
        enlarged_image_grid = sp.resize(sp.ifft(data,axes=(-3, -2, -1)), [n_coils,im_dim, im_dim, im_dim])
        print('fft done input zp')
    else:
        enlarged_image_grid = sp.resize(data, [n_coils,im_dim, im_dim, im_dim])
    if output_kspace:
        enlarged_cartesian_kspace = sp.fft(enlarged_image_grid, axes=(-3, -2, -1))
        print('fft done output zp')
        return enlarged_cartesian_kspace
    else:
        return enlarged_image_grid


def zeropad_riesling(data, im_dim, input_kspace=True, output_kspace=True):
    n_coils = data.shape[0]
    print('starting resize')
    if input_kspace:
        data = sp.ifft(data,axes=(-3, -2, -1))
        !riesling op-resize 
        #enlarged_image_grid = sp.resize(data, [n_coils,im_dim, im_dim, im_dim])
        print('fft done input zp')
    else:
        enlarged_image_grid = sp.resize(data, [n_coils,im_dim, im_dim, im_dim])
    if output_kspace:
        enlarged_cartesian_kspace = sp.fft(enlarged_image_grid, axes=(-3, -2, -1))
        print('fft done output zp')
        return enlarged_cartesian_kspace
    else:
        return enlarged_image_grid




def inner_portion(enlarged_kspace, inner_sidelen):

    cx, cy, cz = enlarged_kspace.shape[1] // 2, enlarged_kspace.shape[2] // 2, enlarged_kspace.shape[3] // 2
    N = inner_sidelen/2
    start, end = int(cx-N), int(cx+N)
    isolation_mask = np.ones(enlarged_kspace.shape[1:], dtype=int)
    isolation_mask[start:end, start:end, start:end] = 0
    isolated_kspace = enlarged_kspace[:, start:end, start:end, start:end]
    
    return(isolated_kspace, isolation_mask, start, end)

def jigsaw3d(output_kspace, start, end, enlarged_kspace):
    recombined = enlarged_kspace.copy()
    print("copy done")
    recombined[:, start:end, start:end, start:end] = output_kspace
    print("jigsaw doneZ")

    return(recombined)

def rebuild3d(output_kspace, inner_start, inner_end, enlarged_kspace, im_dim, output_kspacearr):
    recombined = jigsaw3d(output_kspace=output_kspace, start=inner_start, end=inner_end, enlarged_kspace=enlarged_kspace)
    print("jigsaw3d done")
    filled_ksp = zeropadding3d(recombined, im_dim, output_kspace=output_kspacearr)
    return(filled_ksp)

In [ ]:
def softimpute_ALS_time(X_H, M_H, rank, lamda, n_iters, seed):
    I = np.eye(rank)
    m,n = np.shape(X_H)
    rng = np.random.default_rng(seed)
    U = rng.standard_normal((m, rank)) + 1j * rng.standard_normal((m, rank))
    V = np.zeros((n, rank),dtype=np.complex64)
    U, _ = np.linalg.qr(U)
    D = I.copy()

    A = np.dot(U, D)
    B = np.dot(V, D)
    iter_count = 0
    ABt = A @ B.conj().T
    iter_times = []
    t_total_start = time.perf_counter()
        
    while iter_count < n_iters:
        t_start = time.perf_counter()
        X_star = np.where(M_H, X_H, ABt)
        B = X_star.conj().T @ A @ np.linalg.inv(A.conj().T @ A + lamda*I)
        ABt = A @ B.conj().T

        X_star = np.where(M_H, X_H, ABt)
        A = X_star @ B @ np.linalg.inv(B.conj().T @ B + lamda*I)
        ABt = A @ B.conj().T
        iter_count += 1
        t_elapsed = time.perf_counter() - t_start
        iter_times.append(t_elapsed)

    t_total = time.perf_counter() - t_total_start
    t_mean = np.mean(iter_times)

    return (ABt, t_total, t_mean, iter_times)



def softimpute_ALS(X_H, M_H, rank, lamda, n_iters, seed):
        I = np.eye(rank)
        m,n = np.shape(X_H)
        rng = np.random.default_rng(seed)
        U = rng.standard_normal((m, rank)) + 1j * rng.standard_normal((m, rank))
        V = rng.standard_normal((n, rank)) + 1j * rng.standard_normal((n, rank))

        D = I.copy()


        A = np.dot(U,D)
        B = np.dot(V,D)
        iter_count = 0
        ABt = A @ B.conj().T
        norms = {'A': [], 'B': [], 'ABt': [], 'X_star': [], 'rel_change': [], 'residual': []}
        while iter_count < n_iters:
            ABt_prev = ABt
            X_star = np.where(M_H, X_H, ABt)
            A = X_star @ B @ np.linalg.inv(B.conj().T @ B + lamda*I)
            ABt = A @ B.conj().T
            X_star = np.where(M_H, X_H, ABt)
            B = X_star.conj().T @ A @ np.linalg.inv(A.conj().T @ A + lamda*I)

            norms['A'].append(np.linalg.norm(A))
            norms['B'].append(np.linalg.norm(B))
            norms['ABt'].append(np.linalg.norm(ABt))
            norms['X_star'].append(np.linalg.norm(X_star))
            norms['rel_change'].append(np.linalg.norm(ABt - ABt_prev) / np.linalg.norm(ABt))
            norms['residual'].append(np.linalg.norm(ABt - X_star))

            ABt = A @ B.conj().T
            iter_count += 1
        return(ABt, norms)





def LORAKS_imputeals(n_iters, window_size, cartesian_inputkspace, dtg_mask, rank, lamda, enlarged_kspace, inner_mask, inner_start, inner_end, im_dim, seed):
    ksp_forhankel = cartesian_inputkspace.copy()
    ksp_forhankel = ksp_forhankel * dtg_mask

    hankel_matrix, *_  = hankel(kspace=ksp_forhankel, w=window_size)
    masked_hankel = np.broadcast_to(dtg_mask, (cartesian_inputkspace.shape))
    masked_hankel, *_ = hankel(kspace=masked_hankel, w=window_size)
    masked_hankel = np.real(masked_hankel) > 0.5

    t_als_start = time.perf_counter()
    filled_hankel, norms = softimpute_ALS(X_H = hankel_matrix, M_H = masked_hankel, rank = rank, lamda = lamda, n_iters=n_iters, seed=seed)
    t_als = time.perf_counter() - t_als_start
    print(f"ALS Done ({t_als:.3f}s)")

    kspace_cart_coils_recon = hankel_H_averaged(filled_hankel, n_coils=ksp_forhankel.shape[0], Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2], Nz=ksp_forhankel.shape[3],w=window_size)

    kspace_cart_coils_recon = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
    filled_ksp = rebuild3d(output_kspace=kspace_cart_coils_recon,inner_mask= inner_mask,inner_start= inner_start,inner_end= inner_end, enlarged_kspace= enlarged_kspace,im_dim=im_dim)

    return (filled_ksp, norms)



def LORAKS_imputeals_partial(n_iters, window_size, cartesian_inputkspace, dtg_mask, rank, lamda, seed):
    ksp_forhankel = cartesian_inputkspace.copy()
    ksp_forhankel = ksp_forhankel * dtg_mask

    hankel_matrix, *_  = hankel(kspace=ksp_forhankel, w=window_size)
    masked_hankel = np.broadcast_to(dtg_mask, (cartesian_inputkspace.shape))
    masked_hankel, *_ = hankel(kspace=masked_hankel, w=window_size)
    masked_hankel = np.real(masked_hankel) > 0.5

    t_als_start = time.perf_counter()
    filled_hankel, norms = softimpute_ALS(X_H = hankel_matrix, M_H = masked_hankel, rank = rank, lamda = lamda, n_iters=n_iters, seed=seed)
    t_als = time.perf_counter() - t_als_start
    print(f"ALS Done ({t_als:.3f}s)")

    kspace_cart_coils_recon = hankel_H_averaged(filled_hankel, n_coils=ksp_forhankel.shape[0], Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2], Nz=ksp_forhankel.shape[3],w=window_size)

    kspace_cart_coils_recon = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
    
    return (kspace_cart_coils_recon, norms)


#filled_ksp = rebuild3d(output_kspace=kspace_cart_coils_recon,inner_start= inner_start,inner_end= inner_end, enlarged_kspace= enlarged_kspace,im_dim=im_dim)


In [ ]:
d = np.load(f"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3d Data/inner_cache.npy.npz")
save_dir = "/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3D Data/Testing_lam"

inner_region, start, end = d['inner_region'], d['start'], d['end']

kspace_cart_coils_recon, norms = LORAKS_imputeals_partial(n_iters=50, window_size=12, cartesian_inputkspace=inner_region,dtg_mask=mask, rank=10, lamda=5*10**-5, seed=42)
np.save(f"{save_dir}/kspace_cart_coils_recon_r{r}_win12.npy", kspace_cart_coils_recon)
np.save(f"{save_dir}/norms_r{r}.npy_win12", norms)

In [ ]:
cy = cx = cz = inner_wid // 2
yy, xx, zz = np.ogrid[:inner_wid, :inner_wid, :inner_wid]
dist2 = (yy - cy)**2 + (xx - cx)**2 + (zz - cz)**2

masks = {r: dist2 > r**2 + 2 for r in [2, 3, 4]}

In [ ]:
d = np.load(f"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3d Data/inner_cache.npy.npz")
inner_region, start, end = d['inner_region'], d['start'], d['end']




save_dir = "/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3D Data/Testing_lam"

timings = {'als': []}
recons = {}

for r, mask in masks.items():
    t0 = time.perf_counter()
    kspace_cart_coils_recon, norms = LORAKS_imputeals_partial(
        n_iters=50, window_size=10, cartesian_inputkspace=inner_region,
        dtg_mask=mask, rank=10, lamda=5*10**-5, seed=42
    )
    timings['als'].append(time.perf_counter() - t0)
    recons[r] = (kspace_cart_coils_recon, norms)

    np.save(f"{save_dir}/kspace_cart_coils_recon_r{r}.npy", kspace_cart_coils_recon)
    np.save(f"{save_dir}/norms_r{r}.npy", norms)



In [ ]:
enlarged_cartesian2 = np.load(f"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3d Data/Testing_lam/kspace_cart_coils_recon_r2.npy")
enlarged_cartesian3 = np.load(f"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3d Data/Testing_lam/kspace_cart_coils_recon_r3.npy")
enlarged_cartesian4 = np.load(f"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3d Data/Testing_lam/kspace_cart_coils_recon_r4.npy")
#filled_ksp = rebuild3d(output_kspace=kspace_cart_coils_recon,inner_start= start,inner_end= end, enlarged_kspace= enlarged_cartesian,im_dim=126,output_kspacearr=False)


#del enlarged_cartesian
asm.plot_planes(enlarged_cartesian2, title=None, inputtype_kspace=False)
asm.plot_planes(enlarged_cartesian3, title=None, inputtype_kspace=False)
asm.plot_planes(enlarged_cartesian4, title=None, inputtype_kspace=False)

In [ ]:
save_dir = "/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Results/3D Data/Testing_lam"


kspace_cart_coils_recon = np.load(f"{save_dir}/kspace_cart_coils_recon_r12.npy")
filled_ksp = rebuild3d(output_kspace=kspace_cart_coils_recon, inner_start=start, inner_end=end,enlarged_kspace=enlarged_cartesian, im_dim=126, output_kspacearr=False)

np.save(f"{save_dir}/im_filled_rad{r}.npy", filled_ksp)

In [ ]:
import matplotlib.ticker as mticker

r=3
filled_imgrid = np.load(f"{save_dir}/im_filled_rad{r}.npy")
#riesling_sp_imgrid = np.load(f"{save_dir}/im_filled_rad{r}.npy")
#riesling_sp_imgrid = np.sum(np.abs(filled_imgrid)**2, axis=0)**0.5

recon_rss = np.sum(np.abs(filled_imgrid)**2, axis=0)**0.5
ref_rss   = np.sum(np.abs(riesling_im)**2, axis=0)**0.5
#ref_rss = ref_rss[::-1, ::-1, ::-1]   # flip all three, adjust as needed
diff_rss = (recon_rss - ref_rss) / np.max(np.abs(recon_rss))

nx, ny, nz = recon_rss.shape
slices = lambda v: [
    (v[nx // 2, :, :], 'x mid-slice'),
    (v[:, ny // 2, :], 'y mid-slice'),
    (v[:, :, nz // 2], 'z mid-slice'),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Riesling NUFFT', fontsize=14)
rows = [('recon', recon_rss, 'gray'),
        ('initial',   ref_rss,   'gray'),
        ('diff',  diff_rss,  'RdBu_r')]

for row_axes, (label, vol, cmap) in zip(axes, rows):
    vmax = np.abs(diff_rss).max() if label == 'diff' else None
    for ax, (sl, title) in zip(row_axes, slices(vol)):
        kw = dict(cmap=cmap)
        if label == 'diff':
            kw.update(vmin=-vmax, vmax=vmax)
        im = ax.imshow(sl, **kw)
        ax.set_title(f'{label} {title}')
        ax.axis('off')
        cbar = fig.colorbar(im, ax=ax, fraction=0.046)
        if label == 'diff':
            cbar.formatter = mticker.PercentFormatter(xmax=1.0, decimals=1)
            cbar.update_ticks()
#plt.savefig('riesling_recon.png', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ranks = [2, 3, 4,12]
norms_by_r = {r: np.load(f'{save_dir}/norms_r{r}.npy', allow_pickle=True).item() for r in ranks}

keys = ['A', 'B', 'ABt', 'X_star', 'rel_change', 'residual']

fig, axes = plt.subplots(3, 2, figsize=(15, 10))
for ax, k in zip(axes.flat, keys):
    for r in ranks:
        vals = norms_by_r[r][k]
        iters = range(1, len(vals) + 1)
        ax.plot(iters, vals, marker='o', label=f'r={r}')
    ax.set_title(k)
    ax.set_xlabel('iteration')
    ax.set_ylabel('change (L2 norm)')
    if k == 'rel_change':
        ax.set_yscale('log')
        ax.set_ylabel('relative change')
    ax.grid(True, alpha=0.3, which='both')
    ax.legend()

fig.suptitle('Convergence per iteration')
fig.tight_layout()
plt.show()